# tiny-ton on a Real GPU

This notebook builds **tiny-ton** from source with CUDA enabled and runs `vector_add` on the Colab T4 GPU.

**Before running:** Go to *Runtime > Change runtime type* and select **T4 GPU**.

## 1. Install LLVM/MLIR 18 + pybind11

Uses pre-built packages from `apt.llvm.org` (~2 min).

In [1]:
%%bash
set -e
echo '>>> Adding LLVM 18 apt repository...'
wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key | tee /etc/apt/trusted.gpg.d/apt.llvm.org.asc > /dev/null
add-apt-repository -y 'deb http://apt.llvm.org/jammy/ llvm-toolchain-jammy-18 main' > /dev/null 2>&1
echo '>>> Installing packages...'
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libmlir-18-dev mlir-18-tools llvm-18-dev cmake ninja-build > /dev/null 2>&1
pip install -q pybind11
echo '>>> Done. MLIR version:'
mlir-opt-18 --version 2>&1 | head -2

>>> Adding LLVM 18 apt repository...
>>> Installing packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 9.3 MB/s eta 0:00:00
>>> Done. MLIR version:
Ubuntu LLVM version 18.1.8
  Optimized build.


## 2. Clone tiny-ton

In [2]:
%%bash
set -e
if [ ! -d /content/tiny-ton ]; then
  git clone https://github.com/ganeshnj/tiny-ton.git /content/tiny-ton
else
  echo 'tiny-ton already cloned, pulling latest...'
  cd /content/tiny-ton && git pull
fi

Cloning into '/content/tiny-ton'...


## 3. Build with CUDA enabled

Compiles the C++ compiler, MLIR lowering passes, CUDA runtime, and Python bindings (~1-2 min).

In [3]:
%%bash
set -e
cd /content/tiny-ton
cmake -G Ninja -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \
  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \
  -DTTN_ENABLE_CUDA=ON \
  -DTTN_ENABLE_PYTHON=ON \
  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')
cmake --build build
echo '>>> Build complete.'

-- The CXX compiler identification is GNU 11.4.0
-- The C compiler identification is GNU 11.4.0
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Performing Test HAVE_FFI_CALL
-- Performing Test HAVE_FFI_CALL - Success
-- Found FFI: /usr/lib/x86_64-linux-gnu/libffi.so
-- Could NOT find LibEdit (missing: LibEdit_INCLUDE_DIRS LibEdit_LIBRARIES) 
-- Performing Test Terminfo_LINKABLE
-- Performing Test Terminfo_LINKABLE - Success
-- Found Terminfo: /usr/lib/x86_64-linux-gnu/libtinfo.so
-- Found ZLIB: /usr/lib/x86_64-linux-gnu/libz.so (found version "1.2.11")
-- Found zstd: /usr/lib/x86_64-linux-gnu/libzstd.so
-- Found LibXml2: /usr/li

CMake Error at /usr/local/lib/python3.12/dist-packages/cmake/data/share/cmake-3.31/Modules/FindCUDAToolkit.cmake:908 (message):
  Could not find nvcc, please set CUDAToolkit_ROOT.
Call Stack (most recent call first):
  CMakeLists.txt:34 (find_package)




CalledProcessError: Command 'b"set -e\ncd /content/tiny-ton\ncmake -G Ninja -S . -B build \\\n  -DCMAKE_BUILD_TYPE=Release \\\n  -DMLIR_DIR=/usr/lib/llvm-18/lib/cmake/mlir \\\n  -DLLVM_DIR=/usr/lib/llvm-18/lib/cmake/llvm \\\n  -DTTN_ENABLE_CUDA=ON \\\n  -DTTN_ENABLE_PYTHON=ON \\\n  -Dpybind11_DIR=$(python3 -c 'import pybind11; print(pybind11.get_cmake_dir())')\ncmake --build build\necho '>>> Build complete.'\n"' returned non-zero exit status 1.

## 4. Verify GPU

In [ ]:
!nvidia-smi

import sys
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')

import _tiny_ton_core as core
print(f'\nhas_cuda() = {core.has_cuda()}')

## 5. Debug pipeline (stage-by-stage IR dump)

Shows every compilation stage: Python source -> AST -> TinyTon MLIR -> simulator assembly -> PTX.

In [ ]:
import os
os.environ['TTN_SM_VERSION'] = 'sm_75'

!cd /content/tiny-ton && TTN_SM_VERSION=sm_75 \
  PYTHONPATH=build/bindings:python \
  python3 examples/debug_pipeline.py

## 6. Run vector_add on the real GPU

Full end-to-end: Python -> TinyTon MLIR -> PTX -> CUDA driver API -> T4 GPU -> results back.

In [ ]:
import sys, os
sys.path.insert(0, '/content/tiny-ton/build/bindings')
sys.path.insert(0, '/content/tiny-ton/python')
os.environ['TTN_SM_VERSION'] = 'sm_75'

import numpy as np
import tiny_ton as tt

@tt.jit
def vector_add(a_ptr, b_ptr, c_ptr, N):
    pid = tt.program_id(0)
    offsets = pid * 64 + tt.arange(0, 64)
    mask = offsets < N
    a = tt.load(a_ptr + offsets, mask=mask)
    b = tt.load(b_ptr + offsets, mask=mask)
    tt.store(c_ptr + offsets, a + b, mask=mask)

N = 256
a = np.arange(N, dtype=np.int32)
b = np.arange(N, dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

grid = (N // 64,)
vector_add[grid](a, b, c, N)

print(f'a[:8]      = {a[:8]}')
print(f'b[:8]      = {b[:8]}')
print(f'c[:8]      = {c[:8]}')
print(f'expected   = {(a+b)[:8]}')

assert np.array_equal(c, a + b), 'MISMATCH!'
print('\nPASS -- vector_add ran on the T4 GPU!')

## 7. Run vector_add with f32 on the GPU

Same kernel, float32 arrays -- type is inferred from numpy dtype.

In [ ]:
N = 256
a_f = np.random.randn(N).astype(np.float32)
b_f = np.random.randn(N).astype(np.float32)
c_f = np.zeros(N, dtype=np.float32)

vector_add[(N // 64,)](a_f, b_f, c_f, N)

print(f'a[:8]      = {a_f[:8]}')
print(f'b[:8]      = {b_f[:8]}')
print(f'c[:8]      = {c_f[:8]}')
print(f'expected   = {(a_f+b_f)[:8]}')

assert np.allclose(c_f, a_f + b_f, atol=1e-6), 'f32 MISMATCH!'
print('\nPASS -- f32 vector_add ran on the T4 GPU!')

## 8. Run vector_add with f16 on the GPU

Half-precision float16 -- same kernel, type inferred from `np.float16` arrays. T4 supports native f16 arithmetic (`add.rn.f16` in PTX).

In [ ]:
N = 256
a_h = np.random.randn(N).astype(np.float16)
b_h = np.random.randn(N).astype(np.float16)
c_h = np.zeros(N, dtype=np.float16)

vector_add[(N // 64,)](a_h, b_h, c_h, N)

expected_h = (a_h + b_h).astype(np.float16)
print(f'a[:8]      = {a_h[:8]}')
print(f'b[:8]      = {b_h[:8]}')
print(f'c[:8]      = {c_h[:8]}')
print(f'expected   = {expected_h[:8]}')

assert np.allclose(c_h, expected_h, atol=1e-2), 'f16 MISMATCH!'
print('\nPASS -- f16 vector_add ran on the T4 GPU!')

## 9. Math intrinsics on the GPU

Test `exp`, `log`, `sqrt`, `rsqrt`, `abs`, `max` -- the building blocks of softmax, rmsnorm, and loss.

In [ ]:
@tt.jit
def kern_exp(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.exp(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_log(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.log(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_sqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.sqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_rsqrt(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.rsqrt(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_abs(src, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    tt.store(dst + off, tt.abs(tt.load(src + off, mask=mask)), mask=mask)

@tt.jit
def kern_max(a_ptr, b_ptr, dst, N):
    pid = tt.program_id(0)
    off = pid * 64 + tt.arange(0, 64)
    mask = off < N
    a = tt.load(a_ptr + off, mask=mask)
    b = tt.load(b_ptr + off, mask=mask)
    tt.store(dst + off, tt.max(a, b), mask=mask)

N = 256
grid = (N // 64,)
np.random.seed(42)

for dtype_name, dtype, atol in [('f32', np.float32, 1e-5), ('f16', np.float16, 5e-2)]:
    print(f'\n--- {dtype_name} ---')
    pos = (np.abs(np.random.randn(N)) + 0.01).astype(dtype)
    mixed = (np.random.randn(N) * 2).astype(dtype)

    for name, kernel, np_fn, inp in [
        ('exp',  kern_exp,  np.exp,  mixed),
        ('log',  kern_log,  np.log,  pos),
        ('sqrt', kern_sqrt, np.sqrt, pos),
        ('abs',  kern_abs,  np.abs,  mixed),
    ]:
        dst = np.zeros(N, dtype=dtype)
        kernel[grid](inp.copy(), dst, N)
        ok = np.allclose(dst, np_fn(inp), atol=atol, rtol=1e-2)
        print(f'  {name}: {"PASS" if ok else "FAIL"}')

    # rsqrt
    dst = np.zeros(N, dtype=dtype)
    kern_rsqrt[grid](pos.copy(), dst, N)
    expected_rsqrt = (1.0 / np.sqrt(pos.astype(np.float32))).astype(dtype)
    ok = np.allclose(dst, expected_rsqrt, atol=atol, rtol=1e-2)
    print(f'  rsqrt: {"PASS" if ok else "FAIL"}')

    # max (binary)
    a = (np.random.randn(N) * 5).astype(dtype)
    b = (np.random.randn(N) * 5).astype(dtype)
    dst = np.zeros(N, dtype=dtype)
    kern_max[grid](a.copy(), b.copy(), dst, N)
    ok = np.allclose(dst, np.maximum(a, b), atol=atol, rtol=1e-2)
    print(f'  max: {"PASS" if ok else "FAIL"}')